# Laboratory Day 9a: Semantic Segmentation (U-Net): Data, Tfrecord

In this exercise, we will prepare the data for training a U-Net using Tensorlow. We will learn how to work with "tfrecord", which is an efficient way to store and retrieve data. Before one starts this exercise, it is recommended to read through the tfrecord's tutorial: https://www.tensorflow.org/tutorials/load_data/tfrecord

## 1. Tfrecord (40 points)
**Exercise 1.1 Data Generator (10 points)**:
1. Download the Oxford pet dataset using the following links. Oxford pet dataset: https://www.robots.ox.ac.uk/~vgg/data/pets/

2. Unzip the .tar.gz files. Images are stored under directory "image". Image masks are stored under directory "annotations/trimaps". The dataset provides 3 types of labels: background (label_value=1), pet (label_value=0) and boundary (label_value=2). Images and masks are paired, if they have the same file name, e.g. image="image/Sphynx_103.jpg", mask="annotations/trimaps/Sphynx_103.png"
    
3. In this task, we aim to classify each pixel to "pet" (class=1) and "non-pet" (class=0, includes "background" and "boundary"). Class of "boundary" belong to "non-pet" class. Therefore, if a pixel in directory "annotation/trimaps" has gray value 0, this pixel will be assigned with class 1. Otherwise, this pixel will be assigned with class 0. The total number of classes is 2.

4. Find out the paths of all pairs of images and masks. Separate this dataset into training set and validation set.

5. Write a Python generator function, which yields a pair of image and image mask. Image masks can be still viewed as images, which contain only two types of pixels.

6. Create two generators, one for the training set and the other for the validation set.   

In [ ]:
!curl -O https://thor.robots.ox.ac.uk/~vgg/data/pets/images.tar.gz
!curl -O https://thor.robots.ox.ac.uk/~vgg/data/pets/annotations.tar.gz

In [ ]:
# Solution template
train_paths = [(im1_path, mask1_path), (im2_path, mask2_path), ...]
val_paths = [...]

def data_generator(file_path):
    # file_path: path of image pairs
    # image: [H, W, C=3]
    # image mask: [H, W, C=1]

    # todo
    yield image, image_mask


train_data_gen = data_enerator(train_paths)
val_data_gen = data_enerator(val_paths)


**Exercise 1.2 Data Augmentation (10 points)**

The downloaded dataset contains limited images and image masks. Data augmentation is typically used to improve network accuracy and to avoid overfitting. By using data augmentation, you can add more variety to the training data.

In this exercise, we will implement data argumentation before the training time.

Tasks:

Please write a data augmentation function (a python generator), which randomly adds some (2 to 3 types) of the following effects to the dataset: e.g. Gaussian blurring, Gaussian random noise, random rotation, random crop, random padding, random contrast adjustment. Feel free to add more effects to the dataset. All image libraries, e.g. PIL, scikit-image, can be used.

Please visualize a few examples before and after data augmentation (for quick evaluation of your solution).

Note: If you are not able to finish this task, you could go head and solve the following tasks. However, the training process may be less optimal at the end.


In [ ]:
# Solution
from PIL import Image, ImageFilter #OpenCV can also be used, if needed

def guassian_blurring(im, im_mask):
    #optional to use: ImageFilter.GaussianBlur()
    #todo
    return

def other_op(im, im_mask):
    pass #todo

def data_augmentation(data_generator, num_augmentation=10):
    # data_generator: the generator from the previous exercise
    # num_augmenation: each image is augmented for num_augmenation times

    for image, image_mask in data_generator:
        randomly_selected_ops = [] #todo

        for i in range(num_augmentation):
            for op in randomly_selected_ops:
                image, image_mask = op(image, image_mask)
            yield image, image_mask

**Exercise 1.3 Tfrecord Creation(10 points)**:
1. Based on the developed data generator, convert training and validation data into 2 tfrecord ("train.tfrecord", "val.tfrecord"). 

2. Each record in a tfrecord should contain one image and one image's mask. 

3. Tutorial of tfrecord can be found here: https://www.tensorflow.org/tutorials/load_data/tfrecord#walkthrough_reading_and_writing_image_data

4. Code example to create tfrecord is provided (tfrecord_parse.py). You can use the functions there. 

5. Make sure that the image pair can be converted to tfrecord files and retrieved from tfrecord files in both directions.

6. For task evaluation, Please retrieve the image and the image mask from a generated tfrecord file and visualize 2-3 examples. 

In [ ]:
# Solution template


**Exercise 1.4 Data pipeline (10 points)** :
1. Build data pipelines for the training and validation data, using tf.data.TFRecordDataset(tfrecord_path).

2. For the training dataset, please apply the following operations: (a) repeat the dataset (b) shuffle the dataset (c) convert the record to a pair of image and image mask (d) resize the image and the image mask to 572 * 572 * 3 (e) make up batches with proper batch size.

    - The training dataset is repeated. So we can train the network for any large number of steps.
    
    - 572 * 572 * 3 is the input dimension of U-Net. Optional, if you have memory issues, one could use reduced height and width as the input dimension of U-Net.
    
    - If the images are not with dimension 572*572, they have to be resized during the training time. Optional, one can also resize images during tfrecord generation. Both options are acceptable.

    - If you do not have enough computational power, one could use a reduced input size, e.g. $300*300*3$.

3. For validation dataset, please apply the following operations: (a) convert the record to image and image mask (b) resize the image (c) make up batches.

    - The validation dataset is not repeated. We can evaluate the network's performance regarding the entire validation tfrecord.

4. Test that image and image mask can be retrieved from this data pipeline using e.g. for-loop. Visualize a few examples, which are feed into the network directly.

5. Print out the dimension of the retrieved image and image mask.

In [ ]:
# Solution template
def filter_resize():
# if images are not resized to 572*572

dataset = tf.data.TFRecordDataset(tfrecord_path)
dataset = dataset.repeat()
...
dataset = dataset.map(parser_tfrecord)
dataset = dataset.map(filter_resize)

# add other operations

# test data pipeline
for image, image_mask in dataset:
    # visualize 2-3 images, image_masks